In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from vae_anomaly_detector import (
    ThresholdConfig,
    TrainConfig,
    VAEConfig,
    roc_auc_binary,
    classify,
    find_processed_window_run_by_name,
    load_window_run,
    mse_reconstruction_scores,
    save_training_artifacts,
    select_threshold,
    train_vae,
    binary_metrics,
)

In [2]:
PROCESSED_WINDOWS_ROOT = PROJECT_ROOT / 'dataset' / 'processed_windows'
MODEL_OUTPUT_ROOT = PROJECT_ROOT / 'models' / 'vae_runs'

# Select the processed-window folder by its directory name.
WINDOW_RUN_NAME = '20260422_162223'

selected_window_run = find_processed_window_run_by_name(PROCESSED_WINDOWS_ROOT, WINDOW_RUN_NAME)
print(f'Using processed window run: {selected_window_run}')

Using processed window run: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/dataset/processed_windows/20260422_162223


In [3]:
train_windows, val_windows, test_windows, train_labels, val_labels, test_labels, source_meta = load_window_run(selected_window_run)

# Optional caps to avoid memory/time issues on very large runs.
MAX_TRAIN_WINDOWS = None
MAX_VAL_WINDOWS = None
MAX_TEST_WINDOWS = None

if MAX_TRAIN_WINDOWS is not None and len(train_windows) > MAX_TRAIN_WINDOWS:
    train_windows = train_windows[:MAX_TRAIN_WINDOWS]
    train_labels = train_labels[:MAX_TRAIN_WINDOWS]

if MAX_VAL_WINDOWS is not None and len(val_windows) > MAX_VAL_WINDOWS:
    val_windows = val_windows[:MAX_VAL_WINDOWS]
    val_labels = val_labels[:MAX_VAL_WINDOWS]

if MAX_TEST_WINDOWS is not None and len(test_windows) > MAX_TEST_WINDOWS:
    test_windows = test_windows[:MAX_TEST_WINDOWS]
    test_labels = test_labels[:MAX_TEST_WINDOWS]

print('Loaded arrays:')
print(f' - train_windows: {train_windows.shape}')
print(f' - val_windows: {val_windows.shape}')
print(f' - test_windows:  {test_windows.shape}')
print(f' - train_labels:  {train_labels.shape}, positive={int(train_labels.sum())}')
print(f' - val_labels:  {val_labels.shape}, positive={int(val_labels.sum())}')
print(f' - test_labels:   {test_labels.shape}, positive={int(test_labels.sum())}')

Loaded arrays:
 - train_windows: (445298, 1, 15)
 - val_windows: (198734, 1, 15)
 - test_windows:  (429314, 1, 15)
 - train_labels:  (445298,), positive=0
 - val_labels:  (198734,), positive=8657
 - test_labels:   (429314,), positive=19675


In [4]:
# Core architecture params
vae_cfg = VAEConfig(
    architecture='dense',  # 'dense', 'conv1d', or 'lstm_autoencoder'
    encoder_use_batchnorm=False
)

# Optimization and loss weighting
train_cfg = TrainConfig(
    epochs=10,
    beta=0,
    validation_split=0,
    random_seed=4
)

# Thresholding strategy for anomaly decision
threshold_cfg = ThresholdConfig(
    method='val_f1'
)

vae_cfg, train_cfg, threshold_cfg

(VAEConfig(latent_dim=16, architecture='dense', conv_filters=(32, 64), kernel_size=3, hidden_units=(64, 32), lstm_units=(64, 32), encoder_use_batchnorm=False, encoder_dropout_rate=0.2),
 TrainConfig(epochs=10, batch_size=512, learning_rate=0.001, beta=0, validation_split=0, gradient_clipnorm=1.0, random_seed=4, verbose_epoch=True),
 ThresholdConfig(method='val_f1', percentile=95.0, std_factor=3.0))

In [5]:
encoder, decoder, history = train_vae(
    train_windows,
    vae_cfg,
    train_cfg,
    val_windows=val_windows,
)

history_df = pd.DataFrame(history)
print('Epoch-by-epoch loss report:')
display(history_df)

loss_cols = ['train_total_loss', 'val_total_loss']
non_finite = ~np.isfinite(history_df[loss_cols].to_numpy(dtype=np.float64))
print(f'Any non-finite train/val total loss: {bool(non_finite.any())}')

Epoch 1/10 - train_total=11503.404134, train_recon=11503.404134, train_kl=395069.059084, val_total=9.349261, val_recon=9.349261, val_kl=678.688835
Epoch 2/10 - train_total=0.102711, train_recon=0.102711, train_kl=811.566894, val_total=9.452629, val_recon=9.452629, val_kl=636.026688
Epoch 3/10 - train_total=0.066611, train_recon=0.066611, train_kl=796.761247, val_total=10.097395, val_recon=10.097395, val_kl=637.872550
Epoch 4/10 - train_total=0.056721, train_recon=0.056721, train_kl=784.748271, val_total=9.697121, val_recon=9.697121, val_kl=629.406759
Epoch 5/10 - train_total=0.049574, train_recon=0.049574, train_kl=788.348860, val_total=10.268410, val_recon=10.268410, val_kl=645.094811
Epoch 6/10 - train_total=0.045589, train_recon=0.045589, train_kl=798.654733, val_total=10.250946, val_recon=10.250946, val_kl=641.796548
Epoch 7/10 - train_total=0.039560, train_recon=0.039560, train_kl=787.220449, val_total=9.617612, val_recon=9.617612, val_kl=630.362677
Epoch 8/10 - train_total=0.0366

,epoch,train_total_loss,train_recon_loss,train_kl_loss,val_total_loss,val_recon_loss,val_kl_loss
0,1,11503.404134,11503.404134,395069.059084,9.349261,9.349261,678.688835
1,2,0.102711,0.102711,811.566894,9.452629,9.452629,636.026688
2,3,0.066611,0.066611,796.761247,10.097395,10.097395,637.872550
3,4,0.056721,0.056721,784.748271,9.697121,9.697121,629.406759
4,5,0.049574,0.049574,788.348860,10.268410,10.268410,645.094811
5,6,0.045589,0.045589,798.654733,10.250946,10.250946,641.796548
6,7,0.039560,0.039560,787.220449,9.617612,9.617612,630.362677
7,8,0.036614,0.036614,799.377519,9.776767,9.776767,646.269339
8,9,0.032095,0.032095,787.986702,9.886132,9.886132,622.575543
9,10,0.028719,0.028719,780.098542,10.389094,10.389094,625.750579


Any non-finite train/val total loss: False


In [6]:
# Reconstruction MSE scores (higher means more anomalous)
# Train windows are normal-only by construction from engineer_feature.
train_scores = mse_reconstruction_scores(encoder, decoder, train_windows, batch_size=train_cfg.batch_size)
val_scores = mse_reconstruction_scores(encoder, decoder, val_windows, batch_size=train_cfg.batch_size) if len(val_windows) else np.array([], dtype=np.float32)
test_scores = mse_reconstruction_scores(encoder, decoder, test_windows, batch_size=train_cfg.batch_size)

if threshold_cfg.method == 'val_f1' and len(val_scores) and np.unique(val_labels).size >= 2:
    threshold, threshold_info = select_threshold(
        train_scores=train_scores,
        cfg=threshold_cfg,
        val_scores=val_scores,
        val_labels=val_labels,
    )
else:
    # Fallback when validation labels are empty/single-class.
    fallback_cfg = ThresholdConfig(
        method='percentile',
        percentile=threshold_cfg.percentile,
        std_factor=threshold_cfg.std_factor,
    )
    threshold, threshold_info = select_threshold(train_scores=train_scores, cfg=fallback_cfg)
    if threshold_cfg.method == 'val_f1':
        print('val_f1 fallback -> percentile threshold (validation labels are not suitable).')

train_preds = classify(train_scores, threshold)
test_preds = classify(test_scores, threshold)

print(f'Threshold: {threshold:.6f}')
print(f'Threshold info: {threshold_info}')
print(f'Train anomalies predicted: {int(train_preds.sum())}/{len(train_preds)}')
print(f'Test anomalies predicted: {int(test_preds.sum())}/{len(test_preds)}')

Threshold: 14.926992
Threshold info: {'method': 'val_f1', 'best_val_f1': 0.7410782080485953}
Train anomalies predicted: 22197/445298
Test anomalies predicted: 41892/429314


In [7]:
metrics = binary_metrics(test_labels, test_preds)
metrics['roc_auc'] = roc_auc_binary(test_labels, test_scores)
print('Test metrics (precision, recall, F1, ROC-AUC):')
display(pd.DataFrame([metrics]))

Test metrics (precision, recall, F1, ROC-AUC):


,accuracy,precision,recall,f1,tp,tn,fp,fn,roc_auc
0,0.947141,0.463979,0.987903,0.63141,19437,387184,22455,238,0.991697


In [8]:
saved_run_dir, summary = save_training_artifacts(
    output_root=MODEL_OUTPUT_ROOT,
    encoder=encoder,
    decoder=decoder,
    history=history,
    train_scores=train_scores,
    test_scores=test_scores,
    threshold=threshold,
    train_preds=train_preds,
    test_preds=test_preds,
    test_labels=test_labels,
    vae_cfg=vae_cfg,
    train_cfg=train_cfg,
    threshold_cfg=threshold_cfg,
    source_run_dir=selected_window_run,
 )

print(f'Model artifacts saved to: {saved_run_dir}')
display(pd.json_normalize(summary, sep='.').T.rename(columns={0: 'value'}))

Model artifacts saved to: /Users/Leviathan/Downloads/DDA4210/vae-anomaly-detector/models/vae_runs/20260422_163745


,value
run_id,20260422_163745
source_processed_windows_run,/Users/Leviathan/Downloads/DDA4210/vae-anomaly...
threshold,14.926992
train_samples,445298
test_samples,429314
vae_config.latent_dim,16
vae_config.architecture,dense
vae_config.conv_filters,"(32, 64)"
vae_config.kernel_size,3
vae_config.hidden_units,"(64, 32)"
